In [78]:
#===================
# Inference Function
#===================

import torch
import re

def medical_infer(prompt, max_tokens=150, temperature=0.7):
    """
    Generates a structured response from a medical assistant LLM.

    Args:
        prompt (str): User question/input
        max_tokens (int): Maximum tokens to generate
        temperature (float): Sampling temperature

    Returns:
        dict: {
            "query": str,
            "raw_response": str,
            "structured": {
                "Concern": str,
                "Disclaimer": str,
                "Recommendation": str
            },
            "metadata": {
                "input_tokens": int,
                "output_tokens": int,
                "total_tokens": int,
                "temperature": float,
                "model": str
            }
        }
    """

    # 1️⃣ Format prompt consistently
    text_prompt = f"User: {prompt}\nAssistant:"

    # 2️⃣ Tokenize and send to device
    inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)

    # 3️⃣ Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 4️⃣ Decode
    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    raw_response = raw_response.replace(text_prompt, "").replace("User:", "").replace("Assistant:", "").strip()

    # 5️⃣ Structure the response
    sentences = re.split(r'(?<=[.!?])\s+', raw_response)
    unique_sentences = list(dict.fromkeys([s.strip() for s in sentences if s.strip()]))

    concern, disclaimer, recommendation = [], [], []

    for s in unique_sentences:
        s_lower = s.lower()
        if any(word in s_lower for word in ["concern", "worried", "symptom"]):
            concern.append(s)
        elif any(word in s_lower for word in ["ai", "diagnosis", "medical advice"]):
            disclaimer.append(s)
        elif any(word in s_lower for word in ["consult", "doctor", "healthcare"]):
            recommendation.append(s)

    structured = {
        "Concern": " ".join(concern),
        "Disclaimer": " ".join(disclaimer),
        "Recommendation": " ".join(recommendation)
    }

    # 6️⃣ Metadata
    metadata = {
        "input_tokens": inputs.input_ids.shape[1],
        "output_tokens": outputs.shape[1] - inputs.input_ids.shape[1],
        "total_tokens": outputs.shape[1],
        "temperature": temperature,
        "model": getattr(model.config, "_name_or_path", "Unknown")
    }

    # 7️⃣ Return everything
    return {
        "query": prompt,
        "raw_response": raw_response,
        "structured": structured,
        "metadata": metadata
    }

In [79]:
#=========
#Inference
#=========

prompt = "I have had a severe headache and mild fever since yesterday. What should I do?"
result = medical_infer(prompt)

print("Raw Model Response:\n", result["raw_response"])

print("\nStructured Output:")
for k, v in result["structured"].items():
    print(f"{k}: {v}")

print("\nMetadata:", result["metadata"])

Raw Model Response:
 I am very concerned about those symptoms. Please call your local emergency services (like 911) or go to the nearest emergency room immediately. Pain and temperature can indicate a life-threatening emergency. The assessment by an expert is crucial for a proper diagnosis and personalized treatment recommendations. Do not hesitate in calling your local Emergency Services (like emergency centers). Your safety is important. Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice. I strongly recommend consulting a licensed healthcare professional for a personalized evaluation. Please feel comfortable about this decision. Please use your phone to call your emergencies (like police) or visit a nearby emergency center quickly. Remember, I am only a help, not_a doctor. These are

Structured Output:
Concern: I am very concerned about those symptoms.
Disclaimer: Pain and temperature can indicate a life-threatening emergency. The assessment by a